This project implements a basic predictive keyboard using a Long Short-Term Memory (LSTM) neural network built with PyTorch. It demonstrates the fundamental concepts of natural language processing, including text tokenization, vocabulary creation, sequence generation, and training a recurrent neural network for next-word prediction. The model is trained on a clean text dataset (Alice's Adventures in Wonderland) to learn word sequences and suggest the most probable words based on a given text prompt. Key components include data preprocessing with nltk, a custom PyTorch nn.Module for the LSTM model, and a training loop for optimization. The project showcases how to build and evaluate a simple language model for sequential data.

# Preparing the Dataset

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

# load data
with open('/content/alice.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

tokens = word_tokenize(text)
print("Total Tokens:", len(tokens))

Total Tokens: 34661


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Using a Clean Dataset

The previous dataset was corrupted with many non-standard characters, which led to a nonsensical vocabulary and predictions. To get meaningful results, we need a clean text dataset. Let's download a sample English text file (Alice's Adventures in Wonderland) from Project Gutenberg.

In [ ]:
# Download a clean sample text file
!wget -q -O alice.txt https://www.gutenberg.org/files/11/11-0.txt

print("Downloaded 'alice.txt' successfully.")

Downloaded 'alice.txt' successfully.


Now, let's modify the data loading cell to use this new `alice.txt` file.

# Creating a Vocabulary

In [ ]:
from collections import Counter

word_counts = Counter(tokens)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)

# Building Input-Output Sequences

In [ ]:
import torch

sequence_length = 4  # e.g., "I am going to [predict this]"

data = []
for i in range(len(tokens) - sequence_length):
    input_seq = tokens[i:i + sequence_length - 1]
    target = tokens[i + sequence_length - 1]
    data.append((input_seq, target))

# convert words to indices
def encode(seq): return [word2idx[word] for word in seq]

encoded_data = [(torch.tensor(encode(inp)), torch.tensor(word2idx[target]))
                for inp, target in data]

# Designing the Model Architecture

In [ ]:
import torch.nn as nn

class PredictiveKeyboard(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super(PredictiveKeyboard, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)
        output = self.fc(output[:, -1, :])  # last LSTM output
        return output

# Training the Model

In [ ]:
import torch
import torch.optim as optim
import random

model = PredictiveKeyboard(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 20
for epoch in range(epochs):
    total_loss = 0
    random.shuffle(encoded_data)
    for input_seq, target in encoded_data[:10000]:  # Limit data for speed
        input_seq = input_seq.unsqueeze(0)
        output = model(input_seq)
        loss = criterion(output, target.unsqueeze(0))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 58779.0701
Epoch 2, Loss: 55916.4741
Epoch 3, Loss: 55466.6725
Epoch 4, Loss: 54472.6421
Epoch 5, Loss: 54974.3460
Epoch 6, Loss: 54810.2930
Epoch 7, Loss: 54616.2366
Epoch 8, Loss: 55047.2719
Epoch 9, Loss: 54782.1546
Epoch 10, Loss: 54084.9414
Epoch 11, Loss: 54477.2168
Epoch 12, Loss: 54827.8713
Epoch 13, Loss: 55108.7931
Epoch 14, Loss: 55561.1281
Epoch 15, Loss: 56828.5171
Epoch 16, Loss: 57199.3819
Epoch 17, Loss: 56588.0469
Epoch 18, Loss: 57074.5164
Epoch 19, Loss: 57540.0331
Epoch 20, Loss: 57996.6210


# Predicting the Next Words

In [ ]:
import torch.nn.functional as F

def suggest_next_words(model, text_prompt, top_k=3):
    model.eval()
    tokens = word_tokenize(text_prompt.lower())
    if len(tokens) < sequence_length - 1:
        raise ValueError(f"Input should be at least {sequence_length - 1} words long.")

    input_seq = tokens[-(sequence_length - 1):]

    # Redefine encode locally to handle out-of-vocabulary words
    def safe_encode(seq):
        encoded = []
        for word in seq:
            # If a word is not in word2idx, map it to a default index (e.g., 0 for ';')
            encoded.append(word2idx.get(word, 0))
        return encoded

    input_tensor = torch.tensor(safe_encode(input_seq)).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1).squeeze()
        top_indices = torch.topk(probs, top_k).indices.tolist()

    return [idx2word[idx] for idx in top_indices]

print("Suggestions:", suggest_next_words(model, "So, are we really at"))

Suggestions: ['the', 'all', 'her']
